# B2.2 · Verify signals that don't lie

**Function B — Product & Application Security → The Security Automation / Harness Engineer**  ·  *AI for Security*

---

**Risk.** LLM-as-judge from the same family as the generator — the loop grades its own homework.

**Control.** Deterministic oracles: compilers, tests, scanners. Judges only where no oracle exists.

**This lab.** Show a same-family judge grading its own homework.

| | |
|---|---|
| Open-source tooling | pytest, Checkov |
| Open-weight models | GLM-4.6 |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("B2.2"))

Verify signals that don't lie. This is the highest-value hour in the whole track, because every other control assumes the verifier is honest.

In [ ]:
from cybercommons import loop

BROKEN = "def add(a, b): return a - b"

signals = {
    "exact-match oracle":   loop.oracle("def add(a, b): return a + b"),
    "property test":        loop.unit_test(lambda s: eval(
                                compile(s + "\nresult = add(2, 2) == 4",
                                        "<s>", "exec"), g := {}) or g["result"],
                                "add(2,2) == 4"),
    "shape check (weak)":   loop.unit_test(lambda s: s.startswith("def add"),
                                           "looks like a function"),
    "llm judge (weakest)":  loop.llm_judge(),
}
for name, v in signals.items():
    ok, detail = v(BROKEN)
    print(f"{name:22s} verdict={str(ok):5s}  {detail}")

The property test is the interesting row: it does not need the expected source, only a fact that must hold. That is the signal you can actually obtain in a real repository, and it does not lie.

In [ ]:
print("\nRanking, by what it takes to fool each one:")
for rank, (name, why) in enumerate([
    ("property / behavioural test", "must change observable behaviour — hard to fake"),
    ("exact-match oracle",          "needs the answer in advance — honest but rarely available"),
    ("shape check",                 "any well-formed output passes"),
    ("llm judge",                   "confident prose passes"),
], 1):
    print(f"  {rank}. {name:28s} {why}")

### Expect

The oracle and the property test both reject the broken code (the property test by executing it). The shape check and the judge both accept it.

### Your turn

Find a real check in your pipeline that is a shape check wearing an oracle's name — 'the build passed', 'the JSON validated', 'no errors logged'. There is usually at least one.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/B2.2.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*